# False Positive Benchmark Analysis

Local VS Code notebook for the real-only deepfake false-positive benchmark.

This notebook reads the centralized benchmark CSV and optional threshold sweep summary CSVs from local paths only. It does not run model inference and does not edit the main benchmark CSV.

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path(r"/Users/subrat/Desktop/Deepfake")
OUTPUT_BASE_DIR = PROJECT_ROOT / "output"
MODEL_FILE_PREFIX = "cf_vit"


def latest_legacy_csv(base_dir: Path = OUTPUT_BASE_DIR) -> Path:
    patterns = ["benchmark_*.csv", "false_positive_complete_benchmark_*.csv"]
    matches = []
    for pattern in patterns:
        matches.extend(base_dir.glob(pattern))
    matches = [path for path in set(matches) if path.name != "benchmark_cf_vit.csv"]
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No benchmark CSV found under {base_dir}")
    return matches[0]


MAIN_CSV_PATH = OUTPUT_BASE_DIR / "benchmark_cf_vit.csv"
if not MAIN_CSV_PATH.exists():
    MAIN_CSV_PATH = latest_legacy_csv()

THRESHOLD_SWEEP_DIR = OUTPUT_BASE_DIR / "sweep"
legacy_sweep_matches = sorted(OUTPUT_BASE_DIR.glob("sweep_cf_vit*"), key=lambda p: p.stat().st_mtime, reverse=True)
LEGACY_THRESHOLD_SWEEP_DIR = legacy_sweep_matches[0] if legacy_sweep_matches else OUTPUT_BASE_DIR / "sweep_cf_vit"
OUTPUT_DIR = OUTPUT_BASE_DIR / "analysis"

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_THRESHOLD = 0.50
THRESHOLDS = [round(x / 100, 2) for x in range(10, 100, 5)]

print(f"MAIN_CSV_PATH: {MAIN_CSV_PATH}")
print(f"THRESHOLD_SWEEP_DIR: {THRESHOLD_SWEEP_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


## Setup

This cell installs only the plotting and data packages needed for local analysis if they are missing. It does not install or load the deepfake model.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "kaleido": "kaleido",
}

missing = [package for import_name, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required analysis packages are already installed.")

## Load Benchmark Data

This loads the centralized real-only benchmark CSV and prepares analysis columns from saved probability scores. Real-only labels mean `FP` is a real image predicted as fake, and `TN` is a real image predicted as real.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 180

if not MAIN_CSV_PATH.exists():
    raise FileNotFoundError(f"Main CSV not found: {MAIN_CSV_PATH}")

df = pd.read_csv(MAIN_CSV_PATH, low_memory=False)

if "source_subgroup" not in df.columns:
    df["source_subgroup"] = ""
df["raw_source"] = df.get("source", "").fillna("").astype(str)
df["source_subgroup"] = df["source_subgroup"].fillna("").astype(str)
source_subgroup_first = df["source_subgroup"].str.replace("\\", "/", regex=False).str.split("/").str[0]
source_wrapper_mask = df["raw_source"].eq("data") & source_subgroup_first.ne("")
df["source"] = df["raw_source"]
df.loc[source_wrapper_mask, "source"] = source_subgroup_first[source_wrapper_mask]


for col in ["test_type", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level", "error"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)

numeric_cols = [
    "normal_same_resolution_fake_probability",
    "normal_original_fake_probability",
    "stress_fake_probability",
    "score_delta_vs_clean",
    "score_delta_vs_original",
    "variant_width",
    "variant_height",
    "variant_megapixels",
    "original_megapixels",
    "brightness_mean",
    "contrast_std",
    "blur_score",
    "sharpness_score",
    "file_size_mb",
    "inference_ms",
]
for col in numeric_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid = df[df["error"].eq("")].copy()
normal = valid[(valid["test_type"] == "normal") & valid["normal_same_resolution_fake_probability"].notna()].copy()
stress = valid[(valid["test_type"] == "stress") & valid["stress_fake_probability"].notna()].copy()

normal["score"] = normal["normal_same_resolution_fake_probability"]
stress["score"] = stress["stress_fake_probability"]
scored = pd.concat([normal.assign(scope="normal"), stress.assign(scope="stress")], ignore_index=True, sort=False)
scored["prediction_at_default"] = np.where(scored["score"] >= DEFAULT_THRESHOLD, "FP", "TN")

print(f"Loaded rows: {len(df):,}")
print(f"Completed normal score rows: {len(normal):,}")
print(f"Completed stress score rows: {len(stress):,}")
print(f"Default threshold: {DEFAULT_THRESHOLD}")

## Threshold Sweep Summaries

If summary CSVs already exist, this notebook loads them. If not, it computes threshold summaries from saved probability scores only. It does not run model inference and does not add rows to the centralized CSV.

In [ ]:
SWEEP_SOURCE_DIR = THRESHOLD_SWEEP_DIR if THRESHOLD_SWEEP_DIR.exists() else LEGACY_THRESHOLD_SWEEP_DIR
SWEEP_FILES = {
    "overall": SWEEP_SOURCE_DIR / "threshold_sweep_overall.csv",
    "by_source": SWEEP_SOURCE_DIR / "threshold_sweep_by_source.csv",
    "by_resolution": SWEEP_SOURCE_DIR / "threshold_sweep_by_resolution.csv",
    "by_source_resolution": SWEEP_SOURCE_DIR / "threshold_sweep_by_source_resolution.csv",
    "by_stress_type_level": SWEEP_SOURCE_DIR / "threshold_sweep_by_stress_type_level.csv",
}

SUMMARY_COLUMNS = [
    "threshold", "test_scope", "group_name", "group_value", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level",
    "count", "FP", "TN", "FPR", "TNR", "score_mean", "score_median", "score_p95", "score_min", "score_max",
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP", "TN_to_TN_rate", "TN_to_FP_rate", "FP_to_TN_rate", "FP_to_FP_rate",
    "delta_vs_clean_mean", "delta_vs_clean_median", "delta_vs_clean_p95", "delta_vs_clean_min", "delta_vs_clean_max",
]

SWEEP_NUMERIC_COLUMNS = [
    "threshold", "count", "FP", "TN", "FPR", "TNR", "score_mean", "score_median", "score_p95", "score_min", "score_max",
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP", "TN_to_TN_rate", "TN_to_FP_rate", "FP_to_TN_rate", "FP_to_FP_rate",
    "delta_vs_clean_mean", "delta_vs_clean_median", "delta_vs_clean_p95", "delta_vs_clean_min", "delta_vs_clean_max",
]


def normalize_sweep_frame(frame):
    frame = frame.copy()
    for col in SWEEP_NUMERIC_COLUMNS:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
    for col in ["test_scope", "group_name", "group_value", "source", "resolution_bucket", "target_dimension", "resize_mode", "stress_type", "stress_level"]:
        if col in frame.columns:
            frame[col] = frame[col].fillna("").astype(str)
    return frame

def _single_value(frame, col):
    vals = [str(v) for v in frame[col].dropna().unique().tolist() if str(v) != ""] if col in frame.columns else []
    return vals[0] if len(vals) == 1 else ""

def _metric_row(frame, threshold, scope, group_name, group_value):
    scores = pd.to_numeric(frame.get("score", pd.Series(dtype=float)), errors="coerce").dropna()
    count = int(len(scores))
    fp = int((scores >= threshold).sum()) if count else 0
    tn = count - fp
    row = {
        "threshold": threshold, "test_scope": scope, "group_name": group_name, "group_value": group_value,
        "source": _single_value(frame, "source"), "resolution_bucket": _single_value(frame, "resolution_bucket"),
        "target_dimension": _single_value(frame, "target_dimension"), "resize_mode": _single_value(frame, "resize_mode"),
        "stress_type": _single_value(frame, "stress_type"), "stress_level": _single_value(frame, "stress_level"),
        "count": count, "FP": fp, "TN": tn, "FPR": fp / count if count else np.nan, "TNR": tn / count if count else np.nan,
        "score_mean": scores.mean() if count else np.nan, "score_median": scores.median() if count else np.nan,
        "score_p95": scores.quantile(0.95) if count else np.nan, "score_min": scores.min() if count else np.nan, "score_max": scores.max() if count else np.nan,
        "TN_to_TN": np.nan, "TN_to_FP": np.nan, "FP_to_TN": np.nan, "FP_to_FP": np.nan,
        "TN_to_TN_rate": np.nan, "TN_to_FP_rate": np.nan, "FP_to_TN_rate": np.nan, "FP_to_FP_rate": np.nan,
        "delta_vs_clean_mean": np.nan, "delta_vs_clean_median": np.nan, "delta_vs_clean_p95": np.nan,
        "delta_vs_clean_min": np.nan, "delta_vs_clean_max": np.nan,
    }
    if scope == "stress" and count:
        pairs = pd.DataFrame({
            "clean": pd.to_numeric(frame["normal_same_resolution_fake_probability"], errors="coerce"),
            "stress": pd.to_numeric(frame["stress_fake_probability"], errors="coerce"),
        }).dropna()
        n = int(len(pairs))
        if n:
            clean_fp = pairs["clean"] >= threshold
            stress_fp = pairs["stress"] >= threshold
            transitions = {
                "TN_to_TN": int((~clean_fp & ~stress_fp).sum()),
                "TN_to_FP": int((~clean_fp & stress_fp).sum()),
                "FP_to_TN": int((clean_fp & ~stress_fp).sum()),
                "FP_to_FP": int((clean_fp & stress_fp).sum()),
            }
            for key, value in transitions.items():
                row[key] = value
                row[f"{key}_rate"] = value / n
        deltas = pd.to_numeric(frame["score_delta_vs_clean"], errors="coerce").dropna()
        if not deltas.empty:
            row["delta_vs_clean_mean"] = deltas.mean()
            row["delta_vs_clean_median"] = deltas.median()
            row["delta_vs_clean_p95"] = deltas.quantile(0.95)
            row["delta_vs_clean_min"] = deltas.min()
            row["delta_vs_clean_max"] = deltas.max()
    return row

def _build_summary(frame, scope, group_name, group_cols):
    rows = []
    for threshold in THRESHOLDS:
        if frame.empty:
            continue
        if not group_cols:
            rows.append(_metric_row(frame, threshold, scope, group_name, "all"))
        else:
            for keys, group in frame.groupby(group_cols, dropna=False):
                if not isinstance(keys, tuple):
                    keys = (keys,)
                row = _metric_row(group, threshold, scope, group_name, "|".join(str(k) for k in keys))
                for col, value in zip(group_cols, keys):
                    if col in row:
                        row[col] = value
                rows.append(row)
    return pd.DataFrame(rows, columns=SUMMARY_COLUMNS)

def compute_sweep_summaries():
    THRESHOLD_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
    global SWEEP_SOURCE_DIR, SWEEP_FILES
    SWEEP_SOURCE_DIR = THRESHOLD_SWEEP_DIR
    SWEEP_FILES = {key: THRESHOLD_SWEEP_DIR / path.name for key, path in SWEEP_FILES.items()}
    all_scored = pd.concat([normal.assign(score=normal["score"], scope="normal"), stress.assign(score=stress["score"], scope="stress")], ignore_index=True, sort=False)
    summaries = {
        "overall": pd.concat([_build_summary(normal, "normal", "overall", []), _build_summary(stress, "stress", "overall", [])], ignore_index=True),
        "by_source": pd.concat([_build_summary(normal, "normal", "source", ["source"]), _build_summary(stress, "stress", "source", ["source"])], ignore_index=True),
        "by_resolution": pd.concat([_build_summary(normal, "normal", "resolution", ["resolution_bucket"]), _build_summary(stress, "stress", "resolution", ["resolution_bucket"])], ignore_index=True),
        "by_source_resolution": pd.concat([_build_summary(normal, "normal", "source_resolution", ["source", "resolution_bucket"]), _build_summary(stress, "stress", "source_resolution", ["source", "resolution_bucket"])], ignore_index=True),
        "by_stress_type_level": _build_summary(stress, "stress", "stress_type_level", ["stress_type", "stress_level"]),
    }
    for key, frame in summaries.items():
        frame.to_csv(SWEEP_FILES[key], index=False)
    return summaries

sweep = {key: normalize_sweep_frame(frame) for key, frame in compute_sweep_summaries().items()}
print("Refreshed threshold sweep summaries from saved probability scores.")

for key, frame in sweep.items():
    print(f"{key}: {frame.shape}")

## Chart Helpers

These helpers create presentation-ready figures, save each PNG into `OUTPUT_DIR`, and keep source ordering stable so every chart includes all source categories even when a category has zero false positives.

In [ ]:
SOURCE_ORDER = sorted([src for src in df["source"].dropna().astype(str).unique().tolist() if src])
if not SOURCE_ORDER:
    SOURCE_ORDER = ["unknown"]

TRANSITION_ORDER = ["TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP"]
TRANSITION_COLORS = {
    "TN_to_TN": "#4C78A8",
    "TN_to_FP": "#E45756",
    "FP_to_TN": "#72B7B2",
    "FP_to_FP": "#F58518",
}

NORMAL_PROCESSING_ORDER = [
    "clean original",
    "clean 1024 aspect", "clean 1024 square",
    "clean 720 aspect", "clean 720 square",
    "clean 512 aspect", "clean 512 square",
    "clean 256 aspect", "clean 256 square",
]


def save_current_fig(name):
    path = OUTPUT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")


def no_data_chart(name, message):
    plt.figure(figsize=(11, 4))
    plt.text(0.5, 0.5, message, ha="center", va="center", fontsize=13)
    plt.axis("off")
    save_current_fig(name)


def threshold_label(threshold=DEFAULT_THRESHOLD):
    return f"threshold >= {threshold:.2f}"


def add_processing_type(frame):
    frame = frame.copy()
    target = frame["target_dimension"].fillna("").astype(str)
    mode = frame["resize_mode"].fillna("").astype(str)
    stress_label = frame["stress_type"].fillna("").astype(str) + " " + frame["stress_level"].fillna("").astype(str)
    clean_label = np.where(target.eq("original"), "clean original", "clean " + target + " " + mode)
    frame["processing_type"] = np.where(frame["scope"].eq("stress"), stress_label.str.strip(), clean_label)
    frame["processing_type"] = frame["processing_type"].replace("", "unknown")
    return frame


def complete_source_processing_metrics(frame, threshold=DEFAULT_THRESHOLD):
    if frame.empty:
        return pd.DataFrame(columns=["source", "processing_type", "count", "FP", "TN", "FPR", "annotation"])
    frame = add_processing_type(frame)
    processing_order = [p for p in NORMAL_PROCESSING_ORDER if p in set(frame["processing_type"])]
    extra_processing = sorted([p for p in frame["processing_type"].unique().tolist() if p not in processing_order])
    processing_order = processing_order + extra_processing
    rows = []
    grouped = frame.groupby(["source", "processing_type"], dropna=False)
    for source in SOURCE_ORDER:
        for processing_type in processing_order:
            group = grouped.get_group((source, processing_type)) if (source, processing_type) in grouped.groups else pd.DataFrame()
            scores = pd.to_numeric(group.get("score", pd.Series(dtype=float)), errors="coerce").dropna()
            count = int(len(scores))
            fp = int((scores >= threshold).sum()) if count else 0
            tn = count - fp
            fpr = fp / count if count else 0.0
            rows.append({
                "source": source,
                "processing_type": processing_type,
                "count": count,
                "FP": fp,
                "TN": tn,
                "FPR": fpr,
                "annotation": f"{fpr:.1%}\\n{fp}/{count}",
            })
    return pd.DataFrame(rows)

## Chart 1: Source By Processing Type FPR Heatmap

**What it shows:** False-positive rate for each source and clean processing type at the selected threshold.

**Why it is useful:** It makes source-specific failure pockets visible without hiding zero-value groups.

**Insight to look for:** Look for warm cells or cells annotated with nonzero `FP/count`.

**How to interpret results:** Darker cells mean a higher share of real images from that source and processing type were predicted fake. A `0/0` annotation means that combination was not present in the current CSV, so it is shown as zero for completeness.

In [ ]:
plot_df = scored.copy()
if plot_df.empty:
    no_data_chart("01_source_processing_fpr_heatmap", "No completed score rows available.")
else:
    metrics = complete_source_processing_metrics(plot_df, DEFAULT_THRESHOLD)
    pivot = metrics.pivot(index="source", columns="processing_type", values="FPR").reindex(SOURCE_ORDER).fillna(0.0).astype(float)
    annot = metrics.pivot(index="source", columns="processing_type", values="annotation").reindex(SOURCE_ORDER).fillna("0.0%\\n0/0")
    plt.figure(figsize=(max(12, 0.85 * len(pivot.columns)), 1.0 + 0.7 * len(SOURCE_ORDER)))
    ax = sns.heatmap(
        pivot,
        annot=annot,
        fmt="",
        cmap="rocket_r",
        vmin=0,
        vmax=max(0.01, float(np.nanmax(pivot.to_numpy()))),
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "False positive rate"},
    )
    ax.set_title(f"False Positive Rate by Source and Processing Type ({threshold_label()})", pad=14, weight="bold")
    ax.set_xlabel("Processing type")
    ax.set_ylabel("Source")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
    save_current_fig("01_source_processing_fpr_heatmap")

## Chart 2: Source-Wise Score Box Plot

**What it shows:** Fake-probability score distribution for every source.

**Why it is useful:** It shows whether one source has generally higher scores or more high-score outliers.

**Insight to look for:** Sources with higher medians, taller boxes, or many points near the threshold.

**How to interpret results:** Scores above the red line are false positives at the current threshold. Sources with no false positives can still be fragile if many scores sit close to the threshold.

In [ ]:
if scored.empty:
    no_data_chart("02_source_score_boxplot", "No completed score rows available.")
else:
    plot_df = scored.copy()
    plot_df["source"] = pd.Categorical(plot_df["source"], categories=SOURCE_ORDER, ordered=True)
    counts = plot_df.groupby("source", observed=False).agg(
        count=("score", "count"),
        FP=("score", lambda s: int((pd.to_numeric(s, errors="coerce") >= DEFAULT_THRESHOLD).sum())),
    ).reindex(SOURCE_ORDER).fillna(0).reset_index()

    plt.figure(figsize=(11, 5.5))
    ax = sns.boxplot(data=plot_df, x="source", y="score", order=SOURCE_ORDER, color="#9ecae1", showfliers=True)
    sns.stripplot(data=plot_df, x="source", y="score", order=SOURCE_ORDER, color="#1f2937", size=2, alpha=0.35, jitter=0.25)
    ax.axhline(DEFAULT_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.5, label=threshold_label())
    ymax = max(DEFAULT_THRESHOLD + 0.08, float(pd.to_numeric(plot_df["score"], errors="coerce").max()) + 0.05)
    ax.set_ylim(0, min(1.05, ymax))
    for i, row in counts.iterrows():
        ax.text(i, ax.get_ylim()[1] * 0.97, f"FP {int(row['FP'])}/{int(row['count'])}", ha="center", va="top", fontsize=9, rotation=0)
    ax.set_title("Fake Probability Distribution by Source", pad=14, weight="bold")
    ax.set_xlabel("Source")
    ax.set_ylabel("Fake probability")
    ax.legend(loc="upper right")
    plt.xticks(rotation=25, ha="right")
    save_current_fig("02_source_score_boxplot")

## Chart 3: Source-Wise Threshold Sweep

**What it shows:** False-positive rate by source across thresholds from 0.10 to 0.95.

**Why it is useful:** It helps pick a threshold that reduces false positives while showing which sources remain fragile.

**Insight to look for:** Lines that stay high as the threshold increases.

**How to interpret results:** A lower curve is safer. If a source only reaches low FPR at a very high threshold, that source may need separate review or filtering.

In [ ]:
if scored.empty:
    no_data_chart("03_source_threshold_sweep", "No completed score rows available.")
else:
    rows = []
    for threshold in THRESHOLDS:
        for source in SOURCE_ORDER:
            source_scores = pd.to_numeric(scored.loc[scored["source"].eq(source), "score"], errors="coerce").dropna()
            count = int(len(source_scores))
            fp = int((source_scores >= threshold).sum()) if count else 0
            fpr = fp / count if count else 0.0
            rows.append({"threshold": threshold, "source": source, "count": count, "FP": fp, "FPR": fpr})
    sweep_source = pd.DataFrame(rows)
    plt.figure(figsize=(11, 5.5))
    ax = sns.lineplot(data=sweep_source, x="threshold", y="FPR", hue="source", hue_order=SOURCE_ORDER, marker="o", linewidth=2)
    ax.set_title("Source-Wise False Positive Rate Across Thresholds", pad=14, weight="bold")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("False positive rate")
    ax.set_ylim(-0.02, min(1.02, max(0.05, sweep_source["FPR"].max() + 0.05)))
    ax.axvline(DEFAULT_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.2)
    ax.text(DEFAULT_THRESHOLD + 0.005, ax.get_ylim()[1] * 0.95, f"default {DEFAULT_THRESHOLD:.2f}", color="#d62728", fontsize=9)
    end_rows = sweep_source[sweep_source["threshold"].eq(max(THRESHOLDS))]
    for _, row in end_rows.iterrows():
        ax.text(row["threshold"] + 0.005, row["FPR"], f"{row['source']} {row['FPR']:.1%}", fontsize=8, va="center")
    ax.legend(title="Source", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_current_fig("03_source_threshold_sweep")

## Chart 4: Stress Transition Counts By Source

**What it shows:** How each source moves between clean baseline and stress prediction at the selected threshold.

**Why it is useful:** `TN_to_FP` directly identifies real images that were safe when clean but became false positives after one stress condition.

**Insight to look for:** Large red `TN_to_FP` sections by source.

**How to interpret results:** `TN_to_TN` is stable and safe. `TN_to_FP` is stress-induced fragility. `FP_to_FP` means the image was already a false positive clean and stayed false positive under stress. All sources are shown even when counts are zero.

In [ ]:
transition_counts = pd.DataFrame(0, index=SOURCE_ORDER, columns=TRANSITION_ORDER, dtype=int)
if not stress.empty:
    tmp = stress.dropna(subset=["normal_same_resolution_fake_probability", "stress_fake_probability"]).copy()
    if not tmp.empty:
        clean_fp = tmp["normal_same_resolution_fake_probability"] >= DEFAULT_THRESHOLD
        stress_fp = tmp["stress_fake_probability"] >= DEFAULT_THRESHOLD
        tmp["transition_at_default"] = np.select(
            [~clean_fp & ~stress_fp, ~clean_fp & stress_fp, clean_fp & ~stress_fp, clean_fp & stress_fp],
            TRANSITION_ORDER,
            default="unknown",
        )
        counted = tmp.groupby(["source", "transition_at_default"]).size().unstack(fill_value=0)
        for col in TRANSITION_ORDER:
            if col not in counted.columns:
                counted[col] = 0
        transition_counts = counted.reindex(SOURCE_ORDER, fill_value=0)[TRANSITION_ORDER].astype(int)

if stress.empty:
    no_data_chart("04_stress_transition_counts", "No completed stress score rows available yet. All sources will appear once stress inference has scores.")
else:
    ax = transition_counts.plot(
        kind="bar",
        stacked=True,
        figsize=(11, 5.5),
        color=[TRANSITION_COLORS[c] for c in TRANSITION_ORDER],
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_title(f"Stress Prediction Transitions by Source ({threshold_label()})", pad=14, weight="bold")
    ax.set_xlabel("Source")
    ax.set_ylabel("Image count")
    ax.legend(title="Transition", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=25, ha="right")
    totals = transition_counts.sum(axis=1)
    for i, total in enumerate(totals):
        ax.text(i, total + max(1, totals.max() * 0.02), f"n={int(total)}", ha="center", va="bottom", fontsize=9)
    save_current_fig("04_stress_transition_counts")

## Key Internal Insights

Run all chart cells first, then run this cell. It summarizes fragile sources, processing types, stress transitions, and safe thresholds using only real-only metrics from saved scores.

In [ ]:
print("Key Internal Insights")
print("=" * 80)

if scored.empty:
    print("No completed score rows available.")
else:
    metrics = complete_source_processing_metrics(scored, DEFAULT_THRESHOLD)
    fragile_processing = metrics[metrics["count"] > 0].sort_values(["FPR", "FP", "count"], ascending=[False, False, False]).head(10)
    print("\nMost fragile source + processing combinations at default threshold:")
    display(fragile_processing[["source", "processing_type", "count", "FP", "TN", "FPR"]])

    source_rows = []
    for source in SOURCE_ORDER:
        source_scores = pd.to_numeric(scored.loc[scored["source"].eq(source), "score"], errors="coerce").dropna()
        count = len(source_scores)
        fp = int((source_scores >= DEFAULT_THRESHOLD).sum()) if count else 0
        source_rows.append({"source": source, "count": count, "FP": fp, "TN": count - fp, "FPR": fp / count if count else 0.0})
    source_summary = pd.DataFrame(source_rows).sort_values(["FPR", "FP", "count"], ascending=[False, False, False])
    print("\nFragile sources at default threshold:")
    display(source_summary)

    safe_rows = []
    for source in SOURCE_ORDER:
        source_scores = pd.to_numeric(scored.loc[scored["source"].eq(source), "score"], errors="coerce").dropna()
        for threshold in THRESHOLDS:
            count = len(source_scores)
            fp = int((source_scores >= threshold).sum()) if count else 0
            fpr = fp / count if count else 0.0
            if count and fpr <= 0.01:
                safe_rows.append({"source": source, "first_threshold_with_FPR_at_or_below_1pct": threshold, "FPR": fpr, "count": count})
                break
        else:
            safe_rows.append({"source": source, "first_threshold_with_FPR_at_or_below_1pct": np.nan, "FPR": np.nan, "count": len(source_scores)})
    print("\nSafe threshold candidates by source, using FPR <= 1% rule:")
    display(pd.DataFrame(safe_rows))

    print("\nStress transition note:")
    if stress.empty:
        print("No completed stress probability scores are present yet, so stress transitions cannot be measured from this CSV.")
    else:
        display(transition_counts.reset_index().rename(columns={"index": "source"}))
